In [118]:
import setuptools
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import (
    ensemble,
    preprocessing,
    tree,
)

from sklearn.metrics import (
    auc,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
)

from yellowbrick.classifier import (
    ConfusionMatrix,
    ROCAUC,
)

from yellowbrick.model_selection import (
    LearningCurve,
)   



In [119]:
df_sujo = pd.read_excel("../data/raw/titanic3.xls")
df_testes = pd.read_csv("../data/raw/test.csv")

df_sujo.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [120]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df_sujo, title="Pandas Profiling Report") 
profile.to_file('../data/reports/relatorio_sujo.html')

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 180.49it/s]


In [121]:
df_sujo.dtypes.value_counts()

object     7
int64      4
float64    3
Name: count, dtype: int64

In [122]:
df_sujo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   pclass     1309 non-null   int64  
 1   survived   1309 non-null   int64  
 2   name       1309 non-null   object 
 3   sex        1309 non-null   object 
 4   age        1046 non-null   float64
 5   sibsp      1309 non-null   int64  
 6   parch      1309 non-null   int64  
 7   ticket     1309 non-null   object 
 8   fare       1308 non-null   float64
 9   cabin      295 non-null    object 
 10  embarked   1307 non-null   object 
 11  boat       486 non-null    object 
 12  body       121 non-null    float64
 13  home.dest  745 non-null    object 
dtypes: float64(3), int64(4), object(7)
memory usage: 143.3+ KB


In [123]:
df_sujo.isnull().sum().sort_values(ascending=False)

body         1188
cabin        1014
boat          823
home.dest     564
age           263
embarked        2
fare            1
sibsp           0
name            0
survived        0
pclass          0
sex             0
parch           0
ticket          0
dtype: int64

In [124]:
df_testes.isnull().sum().sort_values(ascending=False)

Cabin          327
Age             86
Fare             1
Name             0
Pclass           0
PassengerId      0
Sex              0
Parch            0
SibSp            0
Ticket           0
Embarked         0
dtype: int64

In [125]:
df = df_sujo.drop(columns=['body', 'cabin', 'boat', 'home.dest', 'name', 'ticket'], axis=1)
df_testes = df_testes.drop(columns=[ 'Cabin', 'Name', 'Ticket'], axis=1)


In [126]:
df.columns = df.columns.str.lower()
df_testes.columns = df_testes.columns.str.lower()

In [127]:
df.loc[df['age'].isnull(), 'age'] = df['age'].mean()
df_testes.loc[df_testes['age'].isnull(), 'age'] = df_testes['age'].mean()

In [128]:
df.embarked.value_counts()

embarked
S    914
C    270
Q    123
Name: count, dtype: int64

In [129]:
df_testes.embarked.value_counts()


embarked
S    270
C    102
Q     46
Name: count, dtype: int64

In [130]:
df.loc[df['embarked'].isnull(), 'embarked'] = df['embarked'].mode()[0]
df_testes.loc[df_testes['embarked'].isnull(), 'embarked'] = df_testes['embarked'].mode()[0]

In [131]:
df.loc[df["fare"].isnull(), "fare"] 

1225   NaN
Name: fare, dtype: float64

In [132]:
df.loc[1225]

pclass         3
survived       0
sex         male
age         60.5
sibsp          0
parch          0
fare         NaN
embarked       S
Name: 1225, dtype: object

In [133]:
df_testes.loc[df_testes["fare"].isnull(), "fare"]  = df_testes["fare"].mean()
df.loc[df["fare"].isnull(), "fare"]  = df["fare"].mean()

In [134]:
df.loc[1225]

pclass              3
survived            0
sex              male
age              60.5
sibsp               0
parch               0
fare        33.295479
embarked            S
Name: 1225, dtype: object

In [135]:
df_testes.isnull().sum().sort_values(ascending=False)


passengerid    0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
dtype: int64

In [136]:
df.isnull().sum().sort_values(ascending=False)

pclass      0
survived    0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64

In [137]:
df_testes.isnull().sum().sort_values(ascending=False)

passengerid    0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
dtype: int64

In [138]:
df.to_csv("../data/interim/titanic_limpo.csv", index=False)
df_testes.to_csv("../data/interim/titanic_testes_limpo.csv", index=False)

In [150]:
treino_nr = df[df.columns[df.dtypes != 'object']]

In [151]:
treino_nr

,pclass,survived,age,sibsp,parch,fare
0,1,1,29.000000,0,0,211.3375
1,1,1,0.916700,1,2,151.5500
2,1,0,2.000000,1,2,151.5500
3,1,0,30.000000,1,2,151.5500
4,1,0,25.000000,1,2,151.5500
...,...,...,...,...,...,...
1304,3,0,14.500000,1,0,14.4542
1305,3,0,29.881135,1,0,14.4542
1306,3,0,26.500000,0,0,7.2250
1307,3,0,27.000000,0,0,7.2250


In [152]:

testes_nr = df_testes[df_testes.columns[df_testes.dtypes != 'object']]

In [153]:
testes_nr

,passengerid,pclass,age,sibsp,parch,fare
0,892,3,34.50000,0,0,7.8292
1,893,3,47.00000,1,0,7.0000
2,894,2,62.00000,0,0,9.6875
3,895,3,27.00000,0,0,8.6625
4,896,3,22.00000,1,1,12.2875
...,...,...,...,...,...,...
413,1305,3,30.27259,0,0,8.0500
414,1306,1,39.00000,0,0,108.9000
415,1307,3,38.50000,0,0,7.2500
416,1308,3,30.27259,0,0,8.0500


In [ ]:
treino_nr.to_csv("../data/interim/titanic_numerical.csv", index=False)
testes_nr.to_csv("../data/interim/titanic_testes_numerical.csv", index=False)